# 02 - CNN Training

This notebook shows the complete classical training workflow for a custom six-class convolutional neural network. Run the cells from top to bottom after preparing the dataset.

## 1. Import Libraries

In [ ]:
from pathlib import Path
import json
import math
import numpy as np
import matplotlib.pyplot as plt
from keras.layers import Conv2D, Dense, Dropout, Flatten, MaxPooling2D
from keras.models import Sequential
from keras.preprocessing.image import ImageDataGenerator
from keras.preprocessing.image import img_to_array, load_img

## 2. Dataset Paths

In [ ]:
DATASET_DIRECTORY = Path('../dataset')
TRAIN_DIRECTORY = DATASET_DIRECTORY / 'train'
VALIDATION_DIRECTORY = DATASET_DIRECTORY / 'validation'
TEST_DIRECTORY = DATASET_DIRECTORY / 'test'
MODEL_DIRECTORY = Path('../model')
OUTPUT_DIRECTORY = Path('../outputs')
IMAGE_SIZE = (150, 150)
BATCH_SIZE = 32
EPOCHS = 20
SEED = 42

for directory in (TRAIN_DIRECTORY, VALIDATION_DIRECTORY, TEST_DIRECTORY):
    if not directory.is_dir():
        raise FileNotFoundError(f'Missing dataset directory: {directory}')

## 3. Image Preprocessing

Every image is resized to 150 × 150 pixels, represented as RGB, and normalized from 0–255 to 0–1. Validation and test images are only normalized.

In [ ]:
validation_data = ImageDataGenerator(rescale=1.0 / 255)
test_data = ImageDataGenerator(rescale=1.0 / 255)

## 4. Data Augmentation

Small geometric variations are applied only to training images to reduce sensitivity to viewpoint and placement.

In [ ]:
training_data = ImageDataGenerator(
    rescale=1.0 / 255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest',
)

training_generator = training_data.flow_from_directory(
    TRAIN_DIRECTORY, target_size=IMAGE_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', shuffle=True, seed=SEED,
)
validation_generator = validation_data.flow_from_directory(
    VALIDATION_DIRECTORY, target_size=IMAGE_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', shuffle=False,
)
test_generator = test_data.flow_from_directory(
    TEST_DIRECTORY, target_size=IMAGE_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', shuffle=False,
)

print('Class indices:', training_generator.class_indices)

## 5. CNN Architecture

The network is trained from scratch. Three convolution and pooling stages learn image features, followed by a dense classifier with dropout.

In [ ]:
model = Sequential([
    Conv2D(32, (3, 3), activation='relu', input_shape=(150, 150, 3)),
    MaxPooling2D(pool_size=(2, 2)),
    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D(pool_size=(2, 2)),
    Conv2D(128, (3, 3), activation='relu'),
    MaxPooling2D(pool_size=(2, 2)),
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(6, activation='softmax'),
])
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

## 6. Model Summary

In [ ]:
model.summary()

## 7. Training

In [ ]:
steps_per_epoch = max(1, int(math.ceil(training_generator.samples / float(BATCH_SIZE))))
validation_steps = max(1, int(math.ceil(validation_generator.samples / float(BATCH_SIZE))))

history = model.fit_generator(
    training_generator,
    steps_per_epoch=steps_per_epoch,
    epochs=EPOCHS,
    validation_data=validation_generator,
    validation_steps=validation_steps,
)

## 8. Accuracy Visualization

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(history.history['acc'], label='Training Accuracy')
plt.plot(history.history['val_acc'], label='Validation Accuracy')
plt.title('Training and Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.show()

## 9. Loss Visualization

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Training and Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.show()

## 10. Testing

In [ ]:
test_steps = max(1, int(math.ceil(test_generator.samples / float(BATCH_SIZE))))
test_generator.reset()
test_loss, test_accuracy = model.evaluate_generator(test_generator, steps=test_steps)
print(f'Test loss: {test_loss:.4f}')
print(f'Test accuracy: {test_accuracy:.4f}')

## 11. Example Prediction

Choose a real test image. The output below is calculated by the trained model and is not hardcoded.

In [ ]:
example_path = next(TEST_DIRECTORY.glob('*/*'))
example_image = load_img(example_path, target_size=IMAGE_SIZE, color_mode='rgb')
example_array = img_to_array(example_image).astype('float32') / 255.0
probabilities = model.predict(np.expand_dims(example_array, axis=0), verbose=0)[0]
index_to_class = {index: name for name, index in training_generator.class_indices.items()}
predicted_index = int(np.argmax(probabilities))
print('Image:', example_path)
print('Prediction:', index_to_class[predicted_index].replace('_', ' ').title())
print(f'Confidence: {probabilities[predicted_index] * 100:.2f}%')

## 12. Saving the Model

Save both the HDF5 model and the generator's exact class mapping. Later predictions must use this mapping.

In [ ]:
MODEL_DIRECTORY.mkdir(parents=True, exist_ok=True)
model.save(str(MODEL_DIRECTORY / 'fruit_freshness_cnn.h5'))
with (MODEL_DIRECTORY / 'class_indices.json').open('w', encoding='utf-8') as mapping_file:
    json.dump(training_generator.class_indices, mapping_file, indent=2, sort_keys=True)
print('Model and class mapping saved.')